In [64]:
import tensorflow as tf 
import pandas as pd

PATH = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data"
PATH_test = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.test"
COLUMNS = ['age','workclass','fnlwgt','education','education_num','marital','occupation','relationship','race','sex','capital_gain','capital_loss','hours_week','native_country','label']


In [65]:
df_train = pd.read_csv( PATH, skipinitialspace=True, names= COLUMNS, index_col=False)
df_test = pd.read_csv( PATH_test,skiprows=1, skipinitialspace=True, names= COLUMNS, index_col=False)

print(df_train.shape, df_test.shape)

print(df_train.dtypes)

(32561, 15) (16281, 15)
age               int64
workclass           str
fnlwgt            int64
education           str
education_num     int64
marital             str
occupation          str
relationship        str
race                str
sex                 str
capital_gain      int64
capital_loss      int64
hours_week        int64
native_country      str
label               str
dtype: object


In [66]:
label = {'<=50K': 0, '>50K': 1}
df_train.label = [label[item] for item in df_train.label]
label_t = {'<=50K.': 0, '>50K.': 1}
df_test.label = [label_t[item] for item in df_test.label]

In [67]:
print(df_train["label"].value_counts())
print(df_test["label"].value_counts())

print(df_train.dtypes)

label
0    24720
1     7841
Name: count, dtype: int64
label
0    12435
1     3846
Name: count, dtype: int64
age               int64
workclass           str
fnlwgt            int64
education           str
education_num     int64
marital             str
occupation          str
relationship        str
race                str
sex                 str
capital_gain      int64
capital_loss      int64
hours_week        int64
native_country      str
label             int64
dtype: object


In [68]:
# Add features to the bucket
# Define continuous list

CONT_FEATURES = ['age', 'fnlwgt', 'capital_gain', 'education_num', 'capital_loss', 'hours_week']
# Define the categorial list
CATE_FEATURES = ['workclass', 'education', 'marital', 'occupation', 'relationship', 'race', 'sex', 'native_country']

from pandas.core.util.hashing import hash_pandas_object
# from sqlalchemy.orm import relationship
continuous_features = [tf.feature_column.numeric_column(k) for k in CONT_FEATURES]
relationship = tf.feature_column.categorical_column_with_vocabulary_list('relationship',['Husband','Not-in-family', 'Wife', 'Own-child', 'Unmarried','Other-relative'])
categorical_features = [tf.feature_column.categorical_column_with_hash_bucket(k,hash_bucket_size = 1000) for k in CATE_FEATURES]


In [69]:
import os
model_dir = os.path.abspath('ongoing/train')
model = tf.estimator.LinearClassifier(n_classes=2, model_dir=model_dir, feature_columns=categorical_features + continuous_features)


INFO:tensorflow:Using default config.
INFO:tensorflow:Using config: {'_model_dir': 'e:\\LANGCHAIN\\Deeplearning\\ongoing\\train', '_tf_random_seed': None, '_save_summary_steps': 100, '_save_checkpoints_steps': None, '_save_checkpoints_secs': 600, '_session_config': allow_soft_placement: true
graph_options {
  rewrite_options {
    meta_optimizer_iterations: ONE
  }
}
, '_keep_checkpoint_max': 5, '_keep_checkpoint_every_n_hours': 10000, '_log_step_count_steps': 100, '_train_distribute': None, '_device_fn': None, '_protocol': None, '_eval_distribute': None, '_experimental_distribute': None, '_experimental_max_worker_delay_secs': None, '_session_creation_timeout_secs': 7200, '_checkpoint_save_graph_def': True, '_service': None, '_cluster_spec': ClusterSpec({}), '_task_type': 'worker', '_task_id': 0, '_global_id_in_cluster': 0, '_master': '', '_evaluation_master': '', '_is_chief': True, '_num_ps_replicas': 0, '_num_worker_replicas': 1}


In [70]:
import numpy as np

# Prepare data types required by TensorFlow Estimators
for col in CATE_FEATURES:
    df_train[col] = df_train[col].astype('object')
    df_test[col] = df_test[col].astype('object')

for col in CONT_FEATURES:
    df_train[col] = df_train[col].astype(np.float32)
    df_test[col] = df_test[col].astype(np.float32)

# Input function using tf.data.Dataset for high performance and compatibility
def get_input_fn(df, num_epochs=None, batch_size=128, shuffle=True):
    def input_fn():
        features = {col: df[col].to_numpy() for col in CONT_FEATURES + CATE_FEATURES}
        labels = df['label'].to_numpy(dtype=np.int32)
        dataset = tf.data.Dataset.from_tensor_slices((features, labels))
        if shuffle:
            dataset = dataset.shuffle(buffer_size=len(df))
        if num_epochs:
            dataset = dataset.repeat(num_epochs)
        else:
            dataset = dataset.repeat()
        dataset = dataset.batch(batch_size)
        return dataset
    return input_fn


In [71]:
# Train the LinearClassifier Estimator
model.train(input_fn=get_input_fn(df_train, num_epochs=None, batch_size=128, shuffle=True), steps=500)


INFO:tensorflow:Calling model_fn.
INFO:tensorflow:Done calling model_fn.
INFO:tensorflow:Create CheckpointSaverHook.
INFO:tensorflow:Graph was finalized.
INFO:tensorflow:Restoring parameters from e:\LANGCHAIN\Deeplearning\ongoing\train\model.ckpt-7306
INFO:tensorflow:Running local_init_op.
INFO:tensorflow:Done running local_init_op.
INFO:tensorflow:Calling checkpoint listeners before saving checkpoint 7306...
INFO:tensorflow:Saving checkpoints for 7306 into e:\LANGCHAIN\Deeplearning\ongoing\train\model.ckpt.
INFO:tensorflow:Calling checkpoint listeners after saving checkpoint 7306...
INFO:tensorflow:loss = 21.331306, step = 7306
INFO:tensorflow:global_step/sec: 414.051
INFO:tensorflow:loss = 99.2631, step = 7406 (0.243 sec)
INFO:tensorflow:global_step/sec: 1385
INFO:tensorflow:loss = 247.0173, step = 7506 (0.072 sec)
INFO:tensorflow:global_step/sec: 1155.93
INFO:tensorflow:loss = 191.13916, step = 7606 (0.087 sec)
INFO:tensorflow:global_step/sec: 1036.99
INFO:tensorflow:loss = 15.53064

In [72]:
# Evaluate on the test dataset
results = model.evaluate(input_fn=get_input_fn(df_test, num_epochs=1, batch_size=128, shuffle=False))
print('Test Accuracy:', results['accuracy'])


INFO:tensorflow:Calling model_fn.
INFO:tensorflow:Done calling model_fn.
INFO:tensorflow:Starting evaluation at 2026-09-25T20:17:53
INFO:tensorflow:Graph was finalized.
INFO:tensorflow:Restoring parameters from e:\LANGCHAIN\Deeplearning\ongoing\train\model.ckpt-7806
INFO:tensorflow:Running local_init_op.
INFO:tensorflow:Done running local_init_op.
INFO:tensorflow:Inference Time : 0.92176s
INFO:tensorflow:Finished evaluation at 2026-09-25-20:17:54
INFO:tensorflow:Saving dict for global step 7806: accuracy = 0.79988945, accuracy_baseline = 0.76377374, auc = 0.6124185, auc_precision_recall = 0.4176352, average_loss = 40.7587, global_step = 7806, label/mean = 0.23622628, loss = 40.633553, precision = 0.7237443, prediction/mean = 0.08073947, recall = 0.24726988
INFO:tensorflow:Saving 'checkpoint_path' summary for global step 7806: e:\LANGCHAIN\Deeplearning\ongoing\train\model.ckpt-7806
Test Accuracy: 0.79988945


In [73]:
import numpy as np

FEATURES = ['age', 'workclass', 'fnlwgt', 'education', 'education_num', 'marital', 'occupation', 'relationship', 'race', 'sex', 'capital_gain', 'capital_loss', 'hours_week', 'native_country']
LABEL = 'label'

def get_input_fn(data_set, num_epochs=None, n_batch=128, shuffle=True):
    # Cast string columns to numpy object and numeric to float32 to ensure compatibility with TensorFlow
    x_dict = {}
    for k in FEATURES:
        if k in CATE_FEATURES:
            x_dict[k] = data_set[k].astype(object)
        else:
            x_dict[k] = data_set[k].astype(np.float32)
    return tf.compat.v1.estimator.inputs.pandas_input_fn(
        x=pd.DataFrame(x_dict),
        y=pd.Series(data_set[LABEL].values, dtype=np.int32),
        batch_size=n_batch,
        num_epochs=num_epochs,
        shuffle=shuffle
    )


In [74]:
model.train(input_fn=get_input_fn(df_train,num_epochs=None, n_batch= 128, shuffle=False),steps = 1000)

INFO:tensorflow:Calling model_fn.


INFO:tensorflow:Done calling model_fn.
INFO:tensorflow:Create CheckpointSaverHook.
INFO:tensorflow:Graph was finalized.
INFO:tensorflow:Restoring parameters from e:\LANGCHAIN\Deeplearning\ongoing\train\model.ckpt-7806
INFO:tensorflow:Running local_init_op.
INFO:tensorflow:Done running local_init_op.
INFO:tensorflow:Calling checkpoint listeners before saving checkpoint 7806...
INFO:tensorflow:Saving checkpoints for 7806 into e:\LANGCHAIN\Deeplearning\ongoing\train\model.ckpt.
INFO:tensorflow:Calling checkpoint listeners after saving checkpoint 7806...
INFO:tensorflow:loss = 65.075165, step = 7806
INFO:tensorflow:global_step/sec: 183.892
INFO:tensorflow:loss = 15.541024, step = 7906 (0.547 sec)
INFO:tensorflow:global_step/sec: 358.164
INFO:tensorflow:loss = 36.46089, step = 8006 (0.282 sec)
INFO:tensorflow:global_step/sec: 388.858
INFO:tensorflow:loss = 173.93048, step = 8106 (0.254 sec)
INFO:tensorflow:global_step/sec: 397.484
INFO:tensorflow:loss = 120.56894, step = 8206 (0.253 sec)
IN

In [75]:
model.evaluate(input_fn=get_input_fn(df_test,num_epochs=1,n_batch=128,shuffle=False),steps=1000)

INFO:tensorflow:Calling model_fn.


INFO:tensorflow:Done calling model_fn.
INFO:tensorflow:Starting evaluation at 2026-09-25T20:18:01
INFO:tensorflow:Graph was finalized.
INFO:tensorflow:Restoring parameters from e:\LANGCHAIN\Deeplearning\ongoing\train\model.ckpt-8806
INFO:tensorflow:Running local_init_op.
INFO:tensorflow:Done running local_init_op.
INFO:tensorflow:Evaluation [100/1000]
INFO:tensorflow:Inference Time : 0.81696s
INFO:tensorflow:Finished evaluation at 2026-09-25-20:18:01
INFO:tensorflow:Saving dict for global step 8806: accuracy = 0.748357, accuracy_baseline = 0.76377374, auc = 0.7856715, auc_precision_recall = 0.47060472, average_loss = 11.807413, global_step = 8806, label/mean = 0.23622628, loss = 11.826333, precision = 0.48052755, prediction/mean = 0.3955731, recall = 0.8052522
INFO:tensorflow:Saving 'checkpoint_path' summary for global step 8806: e:\LANGCHAIN\Deeplearning\ongoing\train\model.ckpt-8806


{'accuracy': 0.748357,
 'accuracy_baseline': 0.76377374,
 'auc': 0.7856715,
 'auc_precision_recall': 0.47060472,
 'average_loss': 11.807413,
 'label/mean': 0.23622628,
 'loss': 11.826333,
 'precision': 0.48052755,
 'prediction/mean': 0.3955731,
 'recall': 0.8052522,
 'global_step': 8806}

In [76]:
# salary is 0 at a very young age and then keeps increasing. Close to retirment it again decreases, hence we square age 
def square_var(df_t, df_te, var_name = 'age'):
    df_t['new'] = df_t[var_name].pow(2)
    df_te['new'] = df_te[var_name].pow(2)
    return df_t, df_te 

In [77]:
df_train_new, df_test_new = square_var(df_train , df_test , var_name = 'age')

In [78]:
print(df_train_new.shape, df_test_new.shape)

(32561, 16) (16281, 16)


In [79]:
CONTI_FEATURES_NEW = ['age','fnlwgt','capital_gain','education_num','capital_loss','hours_week','new']
continuous_features_new = [tf.feature_column.numeric_column(k) for k in CONTI_FEATURES_NEW]

In [80]:
import os
model_dir_1 = os.path.abspath('ongoing/train1')
model_1 = tf.estimator.LinearClassifier(
    model_dir = model_dir_1,
    feature_columns = categorical_features + continuous_features_new
)

INFO:tensorflow:Using default config.
INFO:tensorflow:Using config: {'_model_dir': 'e:\\LANGCHAIN\\Deeplearning\\ongoing\\train1', '_tf_random_seed': None, '_save_summary_steps': 100, '_save_checkpoints_steps': None, '_save_checkpoints_secs': 600, '_session_config': allow_soft_placement: true
graph_options {
  rewrite_options {
    meta_optimizer_iterations: ONE
  }
}
, '_keep_checkpoint_max': 5, '_keep_checkpoint_every_n_hours': 10000, '_log_step_count_steps': 100, '_train_distribute': None, '_device_fn': None, '_protocol': None, '_eval_distribute': None, '_experimental_distribute': None, '_experimental_max_worker_delay_secs': None, '_session_creation_timeout_secs': 7200, '_checkpoint_save_graph_def': True, '_service': None, '_cluster_spec': ClusterSpec({}), '_task_type': 'worker', '_task_id': 0, '_global_id_in_cluster': 0, '_master': '', '_evaluation_master': '', '_is_chief': True, '_num_ps_replicas': 0, '_num_worker_replicas': 1}


In [81]:
FEATURES_NEW = ['age', 'workclass', 'fnlwgt', 'education', 'education_num', 'marital', 'occupation', 'relationship', 'race', 'sex', 'capital_gain', 'capital_loss', 'hours_week', 'native_country', 'new']
def get_input_fn(data_set , num_epochs=None , n_batch=128 , shuffle=True):
    x_dict = {}
    for k in FEATURES_NEW:
        if k in CATE_FEATURES:
            x_dict[k] = data_set[k].astype(object)
        else:
            x_dict[k] = data_set[k].astype(np.float32)
    return tf.compat.v1.estimator.inputs.pandas_input_fn(
        x = pd.DataFrame(x_dict),
        y = pd.Series(data_set[LABEL].values, dtype=np.int32),
        batch_size = n_batch,
        num_epochs = num_epochs,
        shuffle = shuffle
    )

In [82]:
model_1.train(input_fn=get_input_fn(df_train, num_epochs=None, n_batch= 128, shuffle=False), steps=1000)

INFO:tensorflow:Calling model_fn.
INFO:tensorflow:Done calling model_fn.
INFO:tensorflow:Create CheckpointSaverHook.
INFO:tensorflow:Graph was finalized.
INFO:tensorflow:Running local_init_op.
INFO:tensorflow:Done running local_init_op.
INFO:tensorflow:Calling checkpoint listeners before saving checkpoint 0...
INFO:tensorflow:Saving checkpoints for 0 into e:\LANGCHAIN\Deeplearning\ongoing\train1\model.ckpt.
INFO:tensorflow:Calling checkpoint listeners after saving checkpoint 0...
INFO:tensorflow:loss = 0.6931472, step = 0
INFO:tensorflow:global_step/sec: 173.865
INFO:tensorflow:loss = 606.06586, step = 100 (0.580 sec)
INFO:tensorflow:global_step/sec: 405.193
INFO:tensorflow:loss = 260.0491, step = 200 (0.245 sec)
INFO:tensorflow:global_step/sec: 462.33
INFO:tensorflow:loss = 838.5449, step = 300 (0.217 sec)
INFO:tensorflow:global_step/sec: 434.783
INFO:tensorflow:loss = 85.81526, step = 400 (0.231 sec)
INFO:tensorflow:global_step/sec: 441.435
INFO:tensorflow:loss = 133.9148, step = 500

In [83]:
model_1.evaluate(input_fn=get_input_fn(df_test, num_epochs=1, n_batch=128, shuffle=False), steps=1000)

INFO:tensorflow:Calling model_fn.


INFO:tensorflow:Done calling model_fn.
INFO:tensorflow:Starting evaluation at 2026-09-25T20:18:08
INFO:tensorflow:Graph was finalized.
INFO:tensorflow:Restoring parameters from e:\LANGCHAIN\Deeplearning\ongoing\train1\model.ckpt-1000
INFO:tensorflow:Running local_init_op.
INFO:tensorflow:Done running local_init_op.
INFO:tensorflow:Evaluation [100/1000]
INFO:tensorflow:Inference Time : 0.86810s
INFO:tensorflow:Finished evaluation at 2026-09-25-20:18:09
INFO:tensorflow:Saving dict for global step 1000: accuracy = 0.7930717, accuracy_baseline = 0.76377374, auc = 0.6087239, auc_precision_recall = 0.3985445, average_loss = 138.9491, global_step = 1000, label/mean = 0.23622628, loss = 138.63202, precision = 0.65868264, prediction/mean = 0.0922161, recall = 0.2574103
INFO:tensorflow:Saving 'checkpoint_path' summary for global step 1000: e:\LANGCHAIN\Deeplearning\ongoing\train1\model.ckpt-1000


{'accuracy': 0.7930717,
 'accuracy_baseline': 0.76377374,
 'auc': 0.6087239,
 'auc_precision_recall': 0.3985445,
 'average_loss': 138.9491,
 'label/mean': 0.23622628,
 'loss': 138.63202,
 'precision': 0.65868264,
 'prediction/mean': 0.0922161,
 'recall': 0.2574103,
 'global_step': 1000}

In [84]:
predictions = list(model_1.predict(input_fn=get_input_fn(df_train_new, num_epochs=1, n_batch=128, shuffle=False)))

INFO:tensorflow:Calling model_fn.
Instructions for updating:
Use tf.keras instead.
Instructions for updating:
Use tf.keras instead.
Instructions for updating:
Use tf.keras instead.
INFO:tensorflow:Done calling model_fn.
INFO:tensorflow:Graph was finalized.
INFO:tensorflow:Restoring parameters from e:\LANGCHAIN\Deeplearning\ongoing\train1\model.ckpt-1000
INFO:tensorflow:Running local_init_op.
INFO:tensorflow:Done running local_init_op.


In [86]:
print(df_test_new.iloc[3])
print(predictions[3])

age                             44.0
workclass                    Private
fnlwgt                      160323.0
education               Some-college
education_num                   10.0
marital           Married-civ-spouse
occupation         Machine-op-inspct
relationship                 Husband
race                           Black
sex                             Male
capital_gain                  7688.0
capital_loss                     0.0
hours_week                      40.0
native_country         United-States
label                              1
new                           1936.0
Name: 3, dtype: object
{'logits': array([-919.61993], dtype=float32), 'logistic': array([0.], dtype=float32), 'probabilities': array([1., 0.], dtype=float32), 'class_ids': array([0], dtype=int64), 'classes': array([b'0'], dtype=object), 'all_class_ids': array([0, 1]), 'all_classes': array([b'0', b'1'], dtype=object)}


In [87]:
print(df_test_new.iloc[1])
print(predictions[1])

age                             38.0
workclass                    Private
fnlwgt                       89814.0
education                    HS-grad
education_num                    9.0
marital           Married-civ-spouse
occupation           Farming-fishing
relationship                 Husband
race                           White
sex                             Male
capital_gain                     0.0
capital_loss                     0.0
hours_week                      50.0
native_country         United-States
label                              0
new                           1444.0
Name: 1, dtype: object
{'logits': array([-256.4616], dtype=float32), 'logistic': array([0.], dtype=float32), 'probabilities': array([1., 0.], dtype=float32), 'class_ids': array([0], dtype=int64), 'classes': array([b'0'], dtype=object), 'all_class_ids': array([0, 1]), 'all_classes': array([b'0', b'1'], dtype=object)}
